# Eenvoudig klimaatmodel

In dit notebook bouwen we een eenvoudig energiebalansmodel van de aarde in vier stappen:
1. Emissiespectrum van de zon (stralingsintensiteit als functie van golflengte).
2. Gereflecteerde en geabsorbeerde zonnestraling met albedo.
3. Blackbody-uitstraling van de aarde als functie van temperatuur.
4. Evenwichtstemperatuur uit inkomende en uitgaande straling.

We gebruiken een sterk vereenvoudigd model.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Natuurconstanten (SI-eenheden)
h = 6.62607015e-34      # constante van Planck (J s)
c = 2.99792458e8        # lichtsnelheid (m/s)
k_B = 1.380649e-23      # constante van Boltzmann (J/K)
sigma = 5.670374419e-8  # Stefan-Boltzmannconstante (W/m^2/K^4)

# Zon en aarde
T_sun = 5778            # effectieve temperatuur zon (K)
S0 = 1361               # zonneconstante (W/m^2)
albedo = 0.30           # gemiddelde planetaire albedo van de aarde

## 1) Emissiespectrum van de zon
We modelleren de zon als een blackbody met temperatuur $T_{zon}=5778\ \mathrm{K}$.
De spectrale intensiteit (Planck-curve) is:
$$
B_\lambda(T)=\frac{2hc^2}{\lambda^5}\cdot\frac{1}{e^{hc/(\lambda k_B T)}-1}
$$
Hiermee laten we zien bij welke golflengten de zon het meest uitzendt.

In [ ]:
def planck_lambda(wavelength_m, T):
    """Spectrale radiantie B_lambda in W sr^-1 m^-3."""
    a = 2 * h * c**2 / wavelength_m**5
    b = h * c / (wavelength_m * k_B * T)
    return a / (np.exp(b) - 1)

# Golflengtes van 0.1 tot 5 micrometer
lam_um = np.linspace(0.1, 5.0, 1200)
lam_m = lam_um * 1e-6
B_sun = planck_lambda(lam_m, T_sun)

plt.figure(figsize=(8, 4.5))
plt.plot(lam_um, B_sun, color='darkorange')
plt.xlabel('Golflengte (micrometer)')
plt.ylabel('Spectrale radiantie B_lambda (W sr^-1 m^-3)')
plt.title('Emissiespectrum van de zon (blackbody, 5778 K)')
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

# Bepaal piekgolflengte
i_max = np.argmax(B_sun)
lam_peak = lam_um[i_max]
print(f'Piek van het zonnespectrum ligt rond {lam_peak:.2f} micrometer.')

## 2) Gereflecteerde en geabsorbeerde energie (albedo)
Niet alle zonnestraling wordt geabsorbeerd: een deel wordt teruggekaatst door wolken, ijs en lichte oppervlakken.
Met albedo $\alpha$ geldt:
$$
P_{in} = (1-\alpha)\cdot\frac{S_0}{4}
$$
Hier is $S_0$ de zonneconstante en de factor $1/4$ komt door bolgeometrie (doorsnede versus boloppervlak).

In [ ]:
# Inkomend vermogen gemiddeld over de hele aarde
P_in_abs = (1 - albedo) * S0 / 4
P_in_ref = albedo * S0 / 4

print(f'Gereflecteerd gemiddeld vermogen: {P_in_ref:.1f} W/m^2')
print(f'Geabsorbeerd gemiddeld vermogen: {P_in_abs:.1f} W/m^2')

labels = ['Gereflecteerd', 'Geabsorbeerd']
values = [P_in_ref, P_in_abs]
colors = ['lightskyblue', 'gold']

plt.figure(figsize=(6, 4.5))
plt.bar(labels, values, color=colors)
plt.ylabel('Gemiddeld vermogen (W/m^2)')
plt.title(f'Energiebalans inkomende zonnestraling (albedo = {albedo:.2f})')
plt.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

## 3) Blackbody-radiation van de aarde
De aarde straalt in eerste benadering als blackbody uit:
$$
P_{uit}=\sigma T^4
$$
waar $\sigma$ de Stefan-Boltzmannconstante is. We bekijken eerst het spectrum in het infrarood voor verschillende temperaturen.

In [ ]:
lam_earth_um = np.linspace(3.0, 40.0, 1500)
lam_earth_m = lam_earth_um * 1e-6

temperaturen = [240, 255, 270, 288]
kleuren = ['navy', 'teal', 'forestgreen', 'firebrick']

plt.figure(figsize=(8, 4.8))
for T, kleur in zip(temperaturen, kleuren):
    B_earth = planck_lambda(lam_earth_m, T)
    plt.plot(lam_earth_um, B_earth, label=f'{T} K', color=kleur)

plt.xlabel('Golflengte (micrometer)')
plt.ylabel('Spectrale radiantie B_lambda (W sr^-1 m^-3)')
plt.title('Infraroodspectrum van de aarde bij verschillende temperaturen')
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

for T in temperaturen:
    P_out = sigma * T**4
    print(f'T = {T} K -> totaal uitgestraald vermogen = {P_out:.1f} W/m^2')

## 4) Evenwichtstemperatuur berekenen
In stralingsevenwicht geldt:
$$
(1-\alpha)\frac{S_0}{4}=\sigma T_{eq}^4
$$
Dus:
$$
T_{eq}=\left(\frac{(1-\alpha)S_0}{4\sigma}\right)^{1/4}
$$
We berekenen dit voor de aarde en bekijken ook hoe $T_{eq}$ verandert met albedo.

In [ ]:
# Evenwichtstemperatuur voor huidige albedo
T_eq = ((1 - albedo) * S0 / (4 * sigma))**0.25
print(f'Evenwichtstemperatuur zonder broeikaseffect: {T_eq:.2f} K')
print(f'Evenwichtstemperatuur zonder broeikaseffect: {T_eq - 273.15:.2f} °C')

# Gevoeligheid voor albedo
albedo_values = np.linspace(0.1, 0.8, 200)
T_values = ((1 - albedo_values) * S0 / (4 * sigma))**0.25

plt.figure(figsize=(8, 4.5))
plt.plot(albedo_values, T_values - 273.15, color='purple')
plt.axvline(albedo, color='gray', linestyle='--', label=f'Huidig albedo = {albedo:.2f}')
plt.xlabel('Albedo')
plt.ylabel('Evenwichtstemperatuur (°C)')
plt.title('Invloed van albedo op de evenwichtstemperatuur')
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()